# 02 — Training & Evaluation

**Project:** Stripped-context A vs trajectory B for next API-call prediction

This notebook trains on 4 sequential API blocks and evaluates after each.

On this API-Bank task, both conditions predict the next API call. `A` removes prior API trace lines from the prompt, while `B` keeps the full trajectory.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# ============================================================
# CONFIGURATION
# ============================================================
CONDITION = "A"   # "A" or "B"
SEED = 42
assert CONDITION in ("A", "B")

In [24]:
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub tqdm

In [25]:
import json
import os
import re
import random
import time
import pickle
import numpy as np
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling, Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Condition: {CONDITION} | Seed: {SEED}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Condition: A | Seed: 42
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


## 1. Load Preprocessed Data

In [26]:
!unzip -o /content/preprocessed_data.zip -d .

with open('preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

blocks = data['blocks']
config = data['config']
MODEL_NAME = config['model_name']
MAX_SEQ_LEN = config['max_seq_len']
NUM_BLOCKS = config['num_blocks']
BASE_EPOCHS = config['base_epochs']

print(f"Model: {MODEL_NAME}")
print(f"Blocks: {NUM_BLOCKS}, Seq len: {MAX_SEQ_LEN}")
print(f"\nBlock sizes:")
for b in blocks:
    print(f"  D{b['block_id']}: {len(b['train_a'])} train, {len(b['eval_a'])} eval")

Archive:  /content/preprocessed_data.zip
  inflating: ./preprocessed.pkl      
  inflating: ./summary.json          
Model: meta-llama/Llama-3.1-8B-Instruct
Blocks: 4, Seq len: 1024

Block sizes:
  D1: 698 train, 175 eval
  D2: 700 train, 175 eval
  D3: 684 train, 171 eval
  D4: 696 train, 175 eval


In [27]:
def get_train_texts(block, condition):
    if condition == 'A':
        return block['train_a'], block['train_a_prompt_lens']
    return block['train_b'], block['train_b_prompt_lens']

def get_eval_texts(block, condition):
    if condition == 'A':
        return block['eval_a'], block['eval_a_prompt_lens']
    return block['eval_b'], block['eval_b_prompt_lens']

def get_train_tokens(block, condition):
    if condition == 'A':
        return block['train_tokens_a']
    return block['train_tokens_b']

def get_epochs(block, condition):
    return BASE_EPOCHS

## 2. Load Model

In [28]:
ATTN_IMPL = "sdpa"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def load_fresh_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPL,
    )
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

model = load_fresh_model()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


## 3. Dataset & Training

In [29]:
class TextDataset(Dataset):
    # Dataset with prompt-masked labels for causal LM training.
    def __init__(self, texts, prompt_lens, tokenizer, max_length):
        self.items = []
        for text, plen in zip(texts, prompt_lens):
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                add_special_tokens=False,
            )
            input_ids = enc['input_ids']
            attention_mask = enc['attention_mask']
            labels = list(input_ids)
            prompt_cutoff = min(plen, len(labels))
            labels[:prompt_cutoff] = [-100] * prompt_cutoff
            self.items.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        pad_id = self.tokenizer.pad_token_id
        batch_input_ids, batch_attention_mask, batch_labels = [], [], []
        for f in features:
            pad_len = max_len - len(f['input_ids'])
            batch_input_ids.append(f['input_ids'] + [pad_id] * pad_len)
            batch_attention_mask.append(f['attention_mask'] + [0] * pad_len)
            batch_labels.append(f['labels'] + [-100] * pad_len)
        return {
            'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(batch_attention_mask, dtype=torch.long),
            'labels': torch.tensor(batch_labels, dtype=torch.long),
        }

In [30]:
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
GRADIENT_CHECKPOINTING = True
LR = 2e-4

def train_on_block(model, texts, prompt_lens, block_name, num_epochs):
    # Train model on one block. Returns (loss, elapsed).
    dataset = TextDataset(texts, prompt_lens, tokenizer, MAX_SEQ_LEN)
    print(f"  Training {len(dataset)} examples, {num_epochs} epochs...")

    args = TrainingArguments(
        output_dir=f"/tmp/ckpt_{block_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        optim="paged_adamw_8bit",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        seed=SEED,
        dataloader_pin_memory=True,
        dataloader_num_workers=2,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    )

    trainer = Trainer(
        model=model, train_dataset=dataset, args=args,
        data_collator=CausalLMCollator(tokenizer),
    )
    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start
    print(f"  {block_name}: loss={result.training_loss:.4f}, time={elapsed:.0f}s")
    return result.training_loss, elapsed

## 4. Evaluation Functions

In [31]:
EVAL_MAX_SAMPLES = 32

def make_eval_subset_indices(n, sample_seed, max_samples=EVAL_MAX_SAMPLES):
    if n <= max_samples:
        return list(range(n))
    rng = random.Random(sample_seed)
    return sorted(rng.sample(range(n), max_samples))

@torch.no_grad()
def evaluate_loss(model, texts, prompt_lens, sample_seed, max_samples=EVAL_MAX_SAMPLES):
    # Compute average loss and perplexity on response tokens only.
    # Use a fixed subset per block so CL comparisons are stable across stages.
    model.eval()
    indices = make_eval_subset_indices(len(texts), sample_seed, max_samples)
    total_loss, total_tokens = 0.0, 0

    for idx in indices:
        enc = tokenizer(
            texts[idx], truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors="pt",
        ).to(model.device)
        labels = enc['input_ids'].clone()
        plen = min(prompt_lens[idx], labels.shape[1])
        labels[0, :plen] = -100
        labels[enc['attention_mask'] == 0] = -100

        outputs = model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            labels=labels,
        )
        n = (labels != -100).sum().item()
        if n > 0:
            total_loss += outputs.loss.item() * n
            total_tokens += n

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    ppl = np.exp(min(avg_loss, 100))
    model.train()
    return avg_loss, ppl


In [32]:
def strip_trajectory_lines(text):
    lines = []
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('API-Request:') or s.startswith('API-Response:'):
            continue
        if 'Received API Response' in line or 'Generate API Request' in line:
            continue
        lines.append(line)
    return '\n'.join(lines).strip()

CALL_RE = re.compile(r'\[\s*([A-Za-z_][A-Za-z0-9_]*)\((.*?)\)\s*\]', re.DOTALL)
PARAM_RE = re.compile(r"(\w+)='([^']*)'")

def build_generation_prompt(entry, condition):
    # Condition-aware eval prompt. Train/test format MUST match per condition.
    # A (post-only): strip trajectory — matches A training data.
    # B (trajectory): keep trajectory — matches B training data.
    # Prior version stripped for both, causing train/test mismatch for B.
    # See PROJECT_INFO.md L213-232 (condition formats) + L345-347 (Hypothesis 1).
    if condition == 'A':
        context = strip_trajectory_lines(entry['input'])
    else:
        context = entry['input']
    messages = [
        {'role': 'system', 'content': config['system_prompt']},
        {'role': 'user', 'content': context},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def parse_api_call(text):
    match = CALL_RE.search(text)
    if not match:
        return None, None
    api_name = match.group(1)
    params = {k: v for k, v in PARAM_RE.findall(match.group(2))}
    return api_name, params


def normalize_params(params):
    return {k.strip().lower(): v.strip().lower() for k, v in params.items()}

@torch.no_grad()
def evaluate_generation(model, entries, condition, sample_seed, max_samples=EVAL_MAX_SAMPLES):
    # Evaluate exact tool-call generation on the condition's own prompt format.
    # Returns (exact_api_name_accuracy, exact_full_call_accuracy).
    model.eval()

    scored_entries = []
    for entry in entries:
        expected_api, expected_params = parse_api_call(entry.get('output', ''))
        if expected_api is None:
            continue
        scored_entries.append((entry, expected_api, normalize_params(expected_params)))

    if len(scored_entries) > max_samples:
        indices = make_eval_subset_indices(len(scored_entries), sample_seed, max_samples)
        scored_entries = [scored_entries[i] for i in indices]

    name_correct, full_correct = 0, 0
    total = len(scored_entries)

    for entry, expected_api, expected_params in scored_entries:
        prompt = build_generation_prompt(entry, condition)
        enc = tokenizer(
            prompt, truncation=True,
            max_length=MAX_SEQ_LEN - 128, return_tensors="pt",
        ).to(model.device)

        gen = model.generate(
            **enc, max_new_tokens=128,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(
            gen[0][enc['input_ids'].shape[1]:], skip_special_tokens=True
        )

        pred_api, pred_params = parse_api_call(generated)
        if pred_api is None:
            continue

        if pred_api.lower() == expected_api.lower():
            name_correct += 1
            if normalize_params(pred_params) == expected_params:
                full_correct += 1

    model.train()
    name_acc = name_correct / total if total > 0 else 0.0
    full_acc = full_correct / total if total > 0 else 0.0
    return name_acc, full_acc

## 5. Zero-Shot Baseline

In [33]:
print("=" * 60)
print("ZERO-SHOT BASELINE")
print("=" * 60)
print(f"Generation eval uses condition-specific prompt format ({CONDITION}).")

zero_shot = {'loss': [], 'ppl': [], 'name_acc': [], 'full_acc': []}

for j in range(NUM_BLOCKS):
    eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
    loss, ppl = evaluate_loss(model, eval_texts, eval_plens, sample_seed=10_000 + j)
    name_acc, full_acc = evaluate_generation(
        model,
        blocks[j]['eval_entries_raw'],
        condition=CONDITION,
        sample_seed=20_000 + j,
    )
    zero_shot['loss'].append(loss)
    zero_shot['ppl'].append(ppl)
    zero_shot['name_acc'].append(name_acc)
    zero_shot['full_acc'].append(full_acc)
    print(f"  D{j+1}: loss={loss:.3f}, ppl={ppl:.1f}, "
          f"name={name_acc:.1%}, full={full_acc:.1%}")

ZERO-SHOT BASELINE
Generation eval uses condition-specific prompt format (A).
  D1: loss=1.545, ppl=4.7, name=21.9%, full=12.5%
  D2: loss=1.867, ppl=6.5, name=28.1%, full=15.6%
  D3: loss=1.805, ppl=6.1, name=9.4%, full=0.0%
  D4: loss=1.864, ppl=6.5, name=12.5%, full=6.2%


## 6. Continual Learning Loop

In [34]:
print(f"\n{'=' * 60}")
print(f"CONTINUAL LEARNING — Condition {CONDITION}, Seed {SEED}")
print(f"{'=' * 60}")

eval_loss_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_ppl_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_full_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
train_losses, train_times, epochs_per_block = [], [], []

CHECKPOINT_DIR = f"checkpoints_{CONDITION}_seed{SEED}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
experiment_start = time.time()

for i in range(NUM_BLOCKS):
    print(f"\n--- Training on D{i+1}/{NUM_BLOCKS} ---")
    train_texts, train_plens = get_train_texts(blocks[i], CONDITION)
    epochs = get_epochs(blocks[i], CONDITION)
    epochs_per_block.append(epochs)

    t_loss, t_time = train_on_block(
        model, train_texts, train_plens,
        block_name=f"{CONDITION}_s{SEED}_D{i+1}",
        num_epochs=epochs,
    )
    train_losses.append(t_loss)
    train_times.append(t_time)

    # Save LoRA adapter per block — enables eval-only reruns without retraining.
    adapter_path = f"{CHECKPOINT_DIR}/adapter_D{i+1}"
    model.save_pretrained(adapter_path)
    print(f"  Adapter saved: {adapter_path}")

    print(f"  Evaluating all blocks...")
    for j in range(NUM_BLOCKS):
        eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
        loss, ppl = evaluate_loss(model, eval_texts, eval_plens, sample_seed=10_000 + j)
        name_acc, full_acc = evaluate_generation(
            model,
            blocks[j]['eval_entries_raw'],
            condition=CONDITION,
            sample_seed=20_000 + j,
        )
        eval_loss_mat[i][j] = loss
        eval_ppl_mat[i][j] = ppl
        eval_acc_mat[i][j] = name_acc
        eval_full_acc_mat[i][j] = full_acc
        tag = "(curr)" if j == i else "(prev)" if j < i else "(fut)"
        print(f"    D{j+1} {tag}: loss={loss:.3f}, ppl={ppl:.1f}, "
              f"name={name_acc:.1%}, full={full_acc:.1%}")

    # Checkpoint metrics after each block
    ckpt = {
        'condition': CONDITION, 'seed': SEED,
        'blocks_trained': i + 1,
        'eval_loss': eval_loss_mat[:i+1].tolist(),
        'eval_ppl': eval_ppl_mat[:i+1].tolist(),
        'eval_acc': eval_acc_mat[:i+1].tolist(),
        'eval_full_acc': eval_full_acc_mat[:i+1].tolist(),
        'train_losses': train_losses, 'train_times': train_times,
    }
    with open(f'{CHECKPOINT_DIR}/after_D{i+1}.json', 'w') as f:
        json.dump(ckpt, f, indent=2)
    print(f"  Metrics checkpoint saved")

total_time = time.time() - experiment_start
print(f"\nTotal: {total_time:.0f}s ({total_time/60:.1f} min)")


CONTINUAL LEARNING — Condition A, Seed 42

--- Training on D1/4 ---


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training 698 examples, 3 epochs...


Step,Training Loss
10,1.087318
20,0.506185
30,0.524921
40,0.452087
50,0.392080
60,0.224050
70,0.233961
80,0.254621
90,0.203215
100,0.099335


  A_s42_D1: loss=0.3229, time=681s


...

  Training 696 examples, 3 epochs...


Step,Training Loss
10,0.600233
20,0.430872
30,0.489016
40,0.503679
50,0.326837
60,0.200195
70,0.222577
80,0.184369
90,0.175987
100,0.082457


  A_s42_D4: loss=0.2611, time=669s


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69f24ff7-0de5215562991c152b66156f;5c9f9083-9bad-4377-b55e-fe554f2586cc)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in meta-llama/Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


  Adapter saved: checkpoints_A_seed42/adapter_D4
  Evaluating all blocks...
    D1 (prev): loss=0.336, ppl=1.4, name=68.8%, full=37.5%
    D2 (prev): loss=0.612, ppl=1.8, name=59.4%, full=46.9%
    D3 (prev): loss=0.601, ppl=1.8, name=50.0%, full=28.1%
    D4 (curr): loss=0.757, ppl=2.1, name=81.2%, full=40.6%
  Metrics checkpoint saved

Total: 3183s (53.0 min)


## 7. Save Final Results

In [35]:
train_tokens_by_block = [get_train_tokens(block, CONDITION) for block in blocks]

results = {
    'condition': CONDITION,
    'seed': SEED,
    'zero_shot': zero_shot,
    'eval_loss': eval_loss_mat.tolist(),
    'eval_ppl': eval_ppl_mat.tolist(),
    'eval_acc': eval_acc_mat.tolist(),
    'eval_full_acc': eval_full_acc_mat.tolist(),
    'train_losses': train_losses,
    'train_times': train_times,
    'train_tokens_by_block': train_tokens_by_block,
    'total_train_tokens': sum(train_tokens_by_block),
    'epochs_per_block': epochs_per_block,
    'total_time': total_time,
    'config': {
        'model': MODEL_NAME,
        'num_blocks': NUM_BLOCKS,
        'base_epochs': BASE_EPOCHS,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
        'effective_batch_size': BATCH_SIZE * GRAD_ACCUM_STEPS,
        'lr': LR,
        'eval_max_samples': EVAL_MAX_SAMPLES,
        'max_seq_len': MAX_SEQ_LEN,
        'lora_r': 32, 'lora_alpha': 64,
        'attn': ATTN_IMPL,
        'precision': 'bf16',
    },
}

output_file = f"results_{CONDITION}_seed{SEED}.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved: {output_file}")

# Persist checkpoints/results to Google Drive when running in Colab.
# Colab's /content filesystem is temporary; without this backup, adapters are lost
# when the runtime disconnects or resets.
try:
    from google.colab import drive
    import shutil
    from pathlib import Path

    drive.mount('/content/drive', force_remount=False)

    checkpoint_dir = f"checkpoints_{CONDITION}_seed{SEED}"
    bundle_base = f"checkpoints_{CONDITION}_seed{SEED}"
    bundle_file = f"{bundle_base}.tar.gz"

    shutil.make_archive(
        base_name=bundle_base,
        format='gztar',
        root_dir='.',
        base_dir=checkpoint_dir,
    )

    drive_dir = Path('/content/drive/MyDrive/590NN_Final_Project')
    drive_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(bundle_file, drive_dir / bundle_file)
    shutil.copy2(output_file, drive_dir / output_file)

    print(f"Checkpoint bundle saved to Drive: {drive_dir / bundle_file}")
    print(f"Result JSON saved to Drive: {drive_dir / output_file}")
except ModuleNotFoundError:
    print("Google Colab not detected; skipping Drive backup.")
except Exception as exc:
    print(f"Drive backup failed: {exc}")
    raise


Results saved: results_A_seed42.json
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint bundle saved to Drive: /content/drive/MyDrive/590NN_Final_Project/checkpoints_A_seed42.tar.gz
Result JSON saved to Drive: /content/drive/MyDrive/590NN_Final_Project/results_A_seed42.json
